# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [1]:
import pandas as pd
from pathlib import Path

In [23]:
data_path = Path("./data/synthetic_fraud_data.csv")
df = pd.read_csv(data_path)

In [24]:
df.shape

(7483766, 24)

In [25]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7483766 entries, 0 to 7483765
Data columns (total 24 columns):
 #   Column               Dtype  
---  ------               -----  
 0   transaction_id       str    
 1   customer_id          str    
 2   card_number          int64  
 3   timestamp            str    
 4   merchant_category    str    
 5   merchant_type        str    
 6   merchant             str    
 7   amount               float64
 8   currency             str    
 9   country              str    
 10  city                 str    
 11  city_size            str    
 12  card_type            str    
 13  card_present         bool   
 14  device               str    
 15  channel              str    
 16  device_fingerprint   str    
 17  ip_address           str    
 18  distance_from_home   int64  
 19  high_risk_merchant   bool   
 20  transaction_hour     int64  
 21  weekend_transaction  bool   
 22  velocity_last_hour   str    
 23  is_fraud             bool   
dtypes: bool(4

In [26]:
df.head()

,transaction_id,customer_id,card_number,timestamp,merchant_category,merchant_type,merchant,amount,currency,country,...,device,channel,device_fingerprint,ip_address,distance_from_home,high_risk_merchant,transaction_hour,weekend_transaction,velocity_last_hour,is_fraud
0,TX_a0ad2a2a,CUST_72886,6646734767813109,2024-09-30 00:00:01.034820+00:00,Restaurant,fast_food,Taco Bell,294.87,GBP,UK,...,iOS App,mobile,e8e6160445c935fd0001501e4cbac8bc,197.153.60.199,0,False,0,False,"{'num_transactions': 1197, 'total_amount': 334...",False
1,TX_3599c101,CUST_70474,376800864692727,2024-09-30 00:00:01.764464+00:00,Entertainment,gaming,Steam,3368.97,BRL,Brazil,...,Edge,web,a73043a57091e775af37f252b3a32af9,208.123.221.203,1,True,0,False,"{'num_transactions': 509, 'total_amount': 2011...",True
2,TX_a9461c6d,CUST_10715,5251909460951913,2024-09-30 00:00:02.273762+00:00,Grocery,physical,Whole Foods,102582.38,JPY,Japan,...,Firefox,web,218864e94ceaa41577d216b149722261,10.194.159.204,0,False,0,False,"{'num_transactions': 332, 'total_amount': 3916...",False
3,TX_7be21fc4,CUST_16193,376079286931183,2024-09-30 00:00:02.297466+00:00,Gas,major,Exxon,630.60,AUD,Australia,...,iOS App,mobile,70423fa3a1e74d01203cf93b51b9631d,17.230.177.225,0,False,0,False,"{'num_transactions': 764, 'total_amount': 2201...",False
4,TX_150f490b,CUST_87572,6172948052178810,2024-09-30 00:00:02.544063+00:00,Healthcare,medical,Medical Center,724949.27,NGN,Nigeria,...,Chrome,web,9880776c7b6038f2af86bd4e18a1b1a4,136.241.219.151,1,False,0,False,"{'num_transactions': 218, 'total_amount': 4827...",True


card_present: Indicates if the card was physically present during the transaction (used in POS transactions).

---

velocity_last_hour:

Dictionary containing metrics on the transaction velocity, including:
- num_transactions: Number of transactions in the last hour for this customer.
- total_amount: Total amount spent in the last hour.
- unique_merchants: Count of unique merchants in the last hour.
- unique_countries: Count of unique countries in the last hour.
- max_single_amount: Maximum single transaction amount in the last hour.

In [48]:
df_not_fraud = df[df['is_fraud'] == False].sample(n=3_000, random_state=42)
df_fraud = df[df['is_fraud'] == True].sample(n=3_000, random_state=42)
df_mini = pd.concat([df_not_fraud, df_fraud]).sample(frac=1, random_state=42).reset_index(drop=True)

In [49]:
import ast

# Sample 7000 random rows for faster processing


# Parse velocity_last_hour dict string into separate columns
velocity_df = df_mini['velocity_last_hour'].apply(ast.literal_eval).apply(pd.Series)

# Rename columns with _last_hour suffix
velocity_df = velocity_df.rename(columns={
    'num_transactions': 'num_transac_last_hour',
    'total_amount': 'total_amount_last_hour',
    'unique_merchants': 'unique_merchants_last_hour',
    'unique_countries': 'unique_countries_last_hour',
    'max_single_amount': 'max_single_amount_last_hour'
})

# Add the new columns to the dataframe and drop the original
df_mini_expanded = pd.concat([df_mini.drop('velocity_last_hour', axis=1), velocity_df], axis=1)

In [50]:
df_mini_cleaned = df_mini_expanded.drop(["customer_id", "card_number", "merchant", "device_fingerprint", "ip_address"], axis=1)
cols = [c for c in df_mini_cleaned.columns if c != 'is_fraud'] + ['is_fraud']
df_mini_cleaned = df_mini_cleaned[cols]

In [51]:
df_mini_cleaned.head()

,transaction_id,timestamp,merchant_category,merchant_type,amount,currency,country,city,city_size,card_type,...,distance_from_home,high_risk_merchant,transaction_hour,weekend_transaction,num_transac_last_hour,total_amount_last_hour,unique_merchants_last_hour,unique_countries_last_hour,max_single_amount_last_hour,is_fraud
0,TX_a7f9ded2,2024-10-17 15:05:39.603135+00:00,Gas,major,441.96,USD,USA,New York,large,Basic Debit,...,1,False,15,False,482.0,1.402270e+07,102.0,12.0,8.366848e+05,False
1,TX_7eef9759,2024-10-12 03:06:53.888974+00:00,Entertainment,gaming,19.11,BRL,Brazil,Unknown City,medium,Platinum Credit,...,1,True,3,True,424.0,1.554976e+07,98.0,12.0,1.563318e+06,True
2,TX_54b9e43b,2024-10-24 16:47:28.341902+00:00,Gas,major,417.04,USD,USA,Philadelphia,medium,Basic Debit,...,0,False,16,False,193.0,3.494386e+06,86.0,12.0,6.687443e+05,False
3,TX_50a0d972,2024-10-04 09:53:28.036775+00:00,Retail,online,669.02,SGD,Singapore,Unknown City,medium,Basic Debit,...,0,False,9,False,1228.0,3.559787e+07,105.0,12.0,2.046594e+06,False
4,TX_f59953ce,2024-10-11 22:14:35.991700+00:00,Gas,local,730115.44,NGN,Nigeria,Unknown City,medium,Basic Credit,...,1,False,22,False,88.0,4.119283e+06,58.0,9.0,1.972915e+06,True


In [52]:
df_mini_cleaned.isna().sum()

transaction_id                 0
timestamp                      0
merchant_category              0
merchant_type                  0
amount                         0
currency                       0
country                        0
city                           0
city_size                      0
card_type                      0
card_present                   0
device                         0
channel                        0
distance_from_home             0
high_risk_merchant             0
transaction_hour               0
weekend_transaction            0
num_transac_last_hour          0
total_amount_last_hour         0
unique_merchants_last_hour     0
unique_countries_last_hour     0
max_single_amount_last_hour    0
is_fraud                       0
dtype: int64

In [53]:
df_mini_cleaned.shape

(6000, 23)

In [56]:
save_path = data_path.parent / "kurbanov_fraud_data.csv"
save_path

PosixPath('data/kurbanov_fraud_data.csv')

In [57]:
df_mini_cleaned.to_csv(save_path)